# Cell 1 — Install and imports

In [1]:
!pip -q install ultralytics pyyaml opencv-python-headless

import os
import gc
import json
import math
import shutil
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from ultralytics import YOLO

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

print("Imports done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Imports done


# Cell 2 — Resolve inputs

In [2]:
PREP_ROOT = "/kaggle/input/datasets/naresh26032005/vinbigdata-prep"

def resolve_comp_root():
    candidates = [
        "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection",
        "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    base = "/kaggle/input"
    for d in os.listdir(base):
        cand = os.path.join(base, d)
        if os.path.exists(os.path.join(cand, "train.csv")) and os.path.exists(os.path.join(cand, "train")):
            return cand
    raise FileNotFoundError("Original competition dataset not found")

COMP_ROOT = resolve_comp_root()
COMP_TRAIN_DIR = os.path.join(COMP_ROOT, "train")
TRAIN_CSV = os.path.join(COMP_ROOT, "train.csv")

cache_df = pd.read_csv(os.path.join(PREP_ROOT, "cache_df.csv"))
train_split = pd.read_csv(os.path.join(PREP_ROOT, "splits", "train_split.csv"))
val_split = pd.read_csv(os.path.join(PREP_ROOT, "splits", "val_split.csv"))

print("PREP_ROOT:", PREP_ROOT)
print("COMP_ROOT:", COMP_ROOT)
print("train rows:", len(train_split), "val rows:", len(val_split))

PREP_ROOT: /kaggle/input/datasets/naresh26032005/vinbigdata-prep
COMP_ROOT: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection
train rows: 12000 val rows: 3000


# Cell 3 — Config

In [3]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

CLASS_NAMES = [
    "Aortic enlargement",
    "Atelectasis",
    "Calcification",
    "Cardiomegaly",
    "Consolidation",
    "ILD",
    "Infiltration",
    "Lung Opacity",
    "Nodule/Mass",
    "Other lesion",
    "Pleural effusion",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis",
]

NO_FINDING_ID = 14

FOCUS_CLASS_IDS = {1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13}
RARE_CLASS_IDS = {2, 8, 9, 11, 12, 13}

YOLO_SIZE = 1024
MODEL_NAME = "yolov8m.pt"

EPOCHS_STAGE1 = 8
BATCH = 4
WORKERS = 2

CACHE_JPEG_QUALITY = 82

WORK_DIR = "/kaggle/working/yolo1024_stage1_bundle"
DATA_DIR = os.path.join(WORK_DIR, "data")
RUN_DIR = os.path.join(WORK_DIR, "runs")

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR, ignore_errors=True)

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Work dir:", WORK_DIR)

Device: 0
Work dir: /kaggle/working/yolo1024_stage1_bundle


# Cell 4 — Helpers

In [4]:
def ensure_uint8(img):
    img = img.astype(np.float32)
    img = img - img.min()
    mx = img.max()
    if mx > 0:
        img = img / mx
    return (img * 255.0).clip(0, 255).astype(np.uint8)

def read_image_any(path):
    ext = Path(path).suffix.lower()
    if ext in [".dcm", ".dicom"]:
        import pydicom
        from pydicom.pixel_data_handlers.util import apply_voi_lut

        ds = pydicom.dcmread(path)
        img = ds.pixel_array
        try:
            img = apply_voi_lut(img, ds)
        except Exception:
            pass

        img = img.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            img = np.max(img) - img

        if np.ptp(img) > 0:
            img = (img - img.min()) / (img.max() - img.min())
        else:
            img = np.zeros_like(img, dtype=np.float32)

        return (img * 255.0).clip(0, 255).astype(np.uint8)

    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Unable to read image: {path}")
    return img

def letterbox(image, new_shape=1024, color=114):
    shape = image.shape[:2]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    ratio = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = (int(round(shape[1] * ratio)), int(round(shape[0] * ratio)))
    dw = new_shape[1] - new_unpad[0]
    dh = new_shape[0] - new_unpad[1]
    dw /= 2
    dh /= 2

    if shape[::-1] != new_unpad:
        image = cv2.resize(image, new_unpad, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))

    image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return image, ratio, (dw, dh)

def xyxy_to_yolo(x1, y1, x2, y2, w, h):
    cx = ((x1 + x2) / 2) / w
    cy = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return cx, cy, bw, bh

def iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter = inter_w * inter_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - inter + 1e-9
    return inter / union

def parse_classes(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [int(x) for x in str(s).split() if str(x).strip().isdigit()]

def get_source_path(image_id, src_path_rel):
    if isinstance(src_path_rel, str) and os.path.exists(src_path_rel):
        return src_path_rel

    for ext in [".dicom", ".dcm", ".png", ".jpg", ".jpeg"]:
        cand = os.path.join(COMP_TRAIN_DIR, f"{image_id}{ext}")
        if os.path.exists(cand):
            return cand

    raise FileNotFoundError(f"Could not locate source file for {image_id}")

def write_sample(src_img, boxes, out_img_path, out_lbl_path):
    img_lb, ratio, (dw, dh) = letterbox(src_img, new_shape=YOLO_SIZE, color=114)
    H, W = img_lb.shape[:2]

    lines = []
    for b in boxes:
        x1, y1, x2, y2 = b["bbox"]
        x1 = x1 * ratio + dw
        x2 = x2 * ratio + dw
        y1 = y1 * ratio + dh
        y2 = y2 * ratio + dh

        x1 = max(0, min(W - 1, x1))
        y1 = max(0, min(H - 1, y1))
        x2 = max(0, min(W - 1, x2))
        y2 = max(0, min(H - 1, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        cx, cy, bw, bh = xyxy_to_yolo(x1, y1, x2, y2, W, H)
        if bw <= 0 or bh <= 0:
            continue

        lines.append(f'{int(b["class_id"])} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

    img_bgr = cv2.cvtColor(img_lb, cv2.COLOR_GRAY2BGR)
    os.makedirs(os.path.dirname(out_img_path), exist_ok=True)
    os.makedirs(os.path.dirname(out_lbl_path), exist_ok=True)

    cv2.imwrite(out_img_path, img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), CACHE_JPEG_QUALITY])
    with open(out_lbl_path, "w") as f:
        f.write("\n".join(lines))

def build_zoom_crop_box(img_shape, boxes, rare=False):
    H, W = img_shape[:2]
    if not boxes:
        return (0, 0, W, H)

    xs1 = [b["bbox"][0] for b in boxes]
    ys1 = [b["bbox"][1] for b in boxes]
    xs2 = [b["bbox"][2] for b in boxes]
    ys2 = [b["bbox"][3] for b in boxes]

    x1 = min(xs1)
    y1 = min(ys1)
    x2 = max(xs2)
    y2 = max(ys2)

    pad = 0.08 if rare else 0.18
    min_frac = 0.25 if rare else 0.35

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    bw = max(x2 - x1, 1.0)
    bh = max(y2 - y1, 1.0)

    new_w = max(bw * (1 + 2 * pad), W * min_frac)
    new_h = max(bh * (1 + 2 * pad), H * min_frac)

    x1 = int(round(cx - new_w / 2))
    y1 = int(round(cy - new_h / 2))
    x2 = int(round(cx + new_w / 2))
    y2 = int(round(cy + new_h / 2))

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(W, x2)
    y2 = min(H, y2)

    if x2 <= x1:
        x2 = min(W, x1 + 2)
    if y2 <= y1:
        y2 = min(H, y1 + 2)

    return (x1, y1, x2, y2)

def transform_boxes_for_crop(boxes, crop_box, min_visibility=0.20):
    cx1, cy1, cx2, cy2 = crop_box
    out = []
    for b in boxes:
        x1, y1, x2, y2 = b["bbox"]
        ix1 = max(x1, cx1)
        iy1 = max(y1, cy1)
        ix2 = min(x2, cx2)
        iy2 = min(y2, cy2)
        if ix2 <= ix1 or iy2 <= iy1:
            continue

        inter_area = (ix2 - ix1) * (iy2 - iy1)
        area = max(1.0, (x2 - x1) * (y2 - y1))
        if inter_area / area < min_visibility:
            continue

        out.append({
            "class_id": int(b["class_id"]),
            "bbox": [ix1 - cx1, iy1 - cy1, ix2 - cx1, iy2 - cy1],
        })
    return out

def merge_boxes_same_class(boxes, iou_thr=0.5, min_area=8.0):
    grouped = defaultdict(list)
    for b in boxes:
        grouped[int(b["class_id"])].append(b)

    merged = []
    for cls_id, items in grouped.items():
        used = [False] * len(items)
        for i in range(len(items)):
            if used[i]:
                continue
            b0 = items[i]["bbox"]
            area0 = max(0, b0[2] - b0[0]) * max(0, b0[3] - b0[1])
            if area0 < min_area:
                used[i] = True
                continue

            cluster = [items[i]]
            used[i] = True
            for j in range(i + 1, len(items)):
                if used[j]:
                    continue
                if iou_xyxy(items[i]["bbox"], items[j]["bbox"]) >= iou_thr:
                    cluster.append(items[j])
                    used[j] = True

            arr = np.array([c["bbox"] for c in cluster], dtype=np.float32)
            merged.append({"class_id": cls_id, "bbox": arr.mean(axis=0).tolist()})
    return merged

print("Helpers ready")

Helpers ready


# Cell 5 — Build annotations

In [5]:
df = pd.read_csv(TRAIN_CSV)

ann_per_image = defaultdict(list)
for _, row in df.iterrows():
    img_id = row["image_id"]
    cls_id = int(row["class_id"])
    if cls_id == NO_FINDING_ID:
        continue
    if pd.isna(row["x_min"]) or pd.isna(row["y_min"]) or pd.isna(row["x_max"]) or pd.isna(row["y_max"]):
        continue
    ann_per_image[img_id].append({
        "class_id": cls_id,
        "bbox": [float(row["x_min"]), float(row["y_min"]), float(row["x_max"]), float(row["y_max"])],
    })

fused_ann_per_image = {img_id: merge_boxes_same_class(boxes, iou_thr=0.5) for img_id, boxes in ann_per_image.items()}
print("Fused annotations ready:", len(fused_ann_per_image))

Fused annotations ready: 4394


# Cell 6 — Class frequency and output folders

In [6]:
train_freq = Counter()
for s in train_split["classes"].tolist():
    for c in parse_classes(s):
        if c != NO_FINDING_ID:
            train_freq[c] += 1

abn_freq_values = [v for k, v in train_freq.items() if v > 0]
base_weight = float(np.median(abn_freq_values)) if len(abn_freq_values) else 1.0

def repeat_full(classes):
    abn = [c for c in classes if c != NO_FINDING_ID]
    if len(abn) == 0:
        return 1
    score = 0.0
    for c in abn:
        score += base_weight / max(train_freq.get(c, 1), 1)
    focus_boost = 1.4 if any(c in FOCUS_CLASS_IDS for c in abn) else 1.0
    return int(np.clip(round(score * focus_boost), 1, 4))

DATA_1024 = os.path.join(DATA_DIR, "yolo1024")
IMG_TRAIN_FULL = os.path.join(DATA_1024, "images", "train", "full")
IMG_TRAIN_ROI = os.path.join(DATA_1024, "images", "train", "roi")
IMG_TRAIN_ROI_TIGHT = os.path.join(DATA_1024, "images", "train", "roi_tight")
IMG_VAL_FULL = os.path.join(DATA_1024, "images", "val", "full")

LBL_TRAIN_FULL = os.path.join(DATA_1024, "labels", "train", "full")
LBL_TRAIN_ROI = os.path.join(DATA_1024, "labels", "train", "roi")
LBL_TRAIN_ROI_TIGHT = os.path.join(DATA_1024, "labels", "train", "roi_tight")
LBL_VAL_FULL = os.path.join(DATA_1024, "labels", "val", "full")

MANIFEST_DIR = os.path.join(DATA_1024, "manifests")

for p in [IMG_TRAIN_FULL, IMG_TRAIN_ROI, IMG_TRAIN_ROI_TIGHT, IMG_VAL_FULL,
          LBL_TRAIN_FULL, LBL_TRAIN_ROI, LBL_TRAIN_ROI_TIGHT, LBL_VAL_FULL, MANIFEST_DIR]:
    os.makedirs(p, exist_ok=True)

print("Output root:", DATA_1024)

Output root: /kaggle/working/yolo1024_stage1_bundle/data/yolo1024


# Cell 7 — Build stage1 dataset

In [7]:
train_records = []
val_records = []

def process_train_row(row):
    image_id = row["image_id"]
    classes = parse_classes(row["classes"])
    src_path = get_source_path(image_id, row["src_path_rel"])

    img = read_image_any(src_path)
    img = ensure_uint8(img)
    boxes = fused_ann_per_image.get(image_id, [])

    # full image
    out_img_full = os.path.join(IMG_TRAIN_FULL, f"{image_id}_full.jpg")
    out_lbl_full = os.path.join(LBL_TRAIN_FULL, f"{image_id}_full.txt")
    write_sample(img, boxes, out_img_full, out_lbl_full)

    train_records.append({
        "image_id": image_id,
        "kind": "full",
        "img_rel_path": os.path.relpath(out_img_full, WORK_DIR),
        "label_rel_path": os.path.relpath(out_lbl_full, WORK_DIR),
        "has_abnormal": int(row["has_abnormal"]),
        "has_no_finding": int(row["has_no_finding"]),
        "classes": row["classes"],
        "repeat": repeat_full(classes),
    })

    if int(row["has_abnormal"]) == 0 or len(boxes) == 0:
        return

    focus = any(c in FOCUS_CLASS_IDS for c in classes)
    rare = any(c in RARE_CLASS_IDS for c in classes)

    # ROI crop
    crop_box = build_zoom_crop_box(img.shape, boxes, rare=rare)
    cropped = img[crop_box[1]:crop_box[3], crop_box[0]:crop_box[2]]
    crop_boxes = transform_boxes_for_crop(boxes, crop_box, min_visibility=0.20 if rare else 0.25)
    if len(crop_boxes) > 0:
        out_img_roi = os.path.join(IMG_TRAIN_ROI, f"{image_id}_roi.jpg")
        out_lbl_roi = os.path.join(LBL_TRAIN_ROI, f"{image_id}_roi.txt")
        write_sample(cropped, crop_boxes, out_img_roi, out_lbl_roi)
        train_records.append({
            "image_id": image_id,
            "kind": "roi",
            "img_rel_path": os.path.relpath(out_img_roi, WORK_DIR),
            "label_rel_path": os.path.relpath(out_lbl_roi, WORK_DIR),
            "has_abnormal": 1,
            "has_no_finding": 0,
            "classes": row["classes"],
            "repeat": 2 if focus else 1,
        })

    # tighter ROI for the difficult classes
    if focus:
        tight_box = build_zoom_crop_box(img.shape, boxes, rare=True)
        tight_crop = img[tight_box[1]:tight_box[3], tight_box[0]:tight_box[2]]
        tight_boxes = transform_boxes_for_crop(boxes, tight_box, min_visibility=0.18)
        if len(tight_boxes) > 0:
            out_img_tight = os.path.join(IMG_TRAIN_ROI_TIGHT, f"{image_id}_roi_tight.jpg")
            out_lbl_tight = os.path.join(LBL_TRAIN_ROI_TIGHT, f"{image_id}_roi_tight.txt")
            write_sample(tight_crop, tight_boxes, out_img_tight, out_lbl_tight)
            train_records.append({
                "image_id": image_id,
                "kind": "roi_tight",
                "img_rel_path": os.path.relpath(out_img_tight, WORK_DIR),
                "label_rel_path": os.path.relpath(out_lbl_tight, WORK_DIR),
                "has_abnormal": 1,
                "has_no_finding": 0,
                "classes": row["classes"],
                "repeat": 2,
            })

def process_val_row(row):
    image_id = row["image_id"]
    src_path = get_source_path(image_id, row["src_path_rel"])
    img = read_image_any(src_path)
    img = ensure_uint8(img)
    boxes = fused_ann_per_image.get(image_id, [])

    out_img_full = os.path.join(IMG_VAL_FULL, f"{image_id}_full.jpg")
    out_lbl_full = os.path.join(LBL_VAL_FULL, f"{image_id}_full.txt")
    write_sample(img, boxes, out_img_full, out_lbl_full)

    val_records.append({
        "image_id": image_id,
        "kind": "full",
        "img_rel_path": os.path.relpath(out_img_full, WORK_DIR),
        "label_rel_path": os.path.relpath(out_lbl_full, WORK_DIR),
        "has_abnormal": int(row["has_abnormal"]),
        "has_no_finding": int(row["has_no_finding"]),
        "classes": row["classes"],
        "repeat": 1,
    })

print("Building stage1 data...")
for _, row in tqdm(train_split.iterrows(), total=len(train_split), desc="train build"):
    process_train_row(row)

for _, row in tqdm(val_split.iterrows(), total=len(val_split), desc="val build"):
    process_val_row(row)

train_records_df = pd.DataFrame(train_records)
val_records_df = pd.DataFrame(val_records)

train_records_df.to_csv(os.path.join(WORK_DIR, "train_records.csv"), index=False)
val_records_df.to_csv(os.path.join(WORK_DIR, "val_records.csv"), index=False)

print("train samples:", len(train_records_df))
print("val samples:", len(val_records_df))

Building stage1 data...


train build:   0%|          | 0/12000 [00:00<?, ?it/s]

val build:   0%|          | 0/3000 [00:00<?, ?it/s]

train samples: 18280
val samples: 3000


# Cell 8 — Write manifests and YAML# 

In [8]:
def write_manifest(paths, out_txt):
    with open(out_txt, "w") as f:
        f.write("\n".join(paths))
    return out_txt

train_paths = []
for _, r in train_records_df.iterrows():
    train_paths.extend([os.path.join(WORK_DIR, r["img_rel_path"])] * int(r["repeat"]))

val_paths = [os.path.join(WORK_DIR, r["img_rel_path"]) for _, r in val_records_df.iterrows()]

TRAIN_TXT = write_manifest(train_paths, os.path.join(MANIFEST_DIR, "train_stage1.txt"))
VAL_TXT = write_manifest(val_paths, os.path.join(MANIFEST_DIR, "val.txt"))

BASE_YAML = os.path.join(DATA_1024, "yolo1024_stage1.yaml")
with open(BASE_YAML, "w") as f:
    f.write(f"path: {DATA_1024}\n")
    f.write(f"train: {TRAIN_TXT}\n")
    f.write(f"val: {VAL_TXT}\n")
    f.write("names:\n")
    for i, name in enumerate(CLASS_NAMES):
        f.write(f"  {i}: {name}\n")

print(open(BASE_YAML).read())

path: /kaggle/working/yolo1024_stage1_bundle/data/yolo1024
train: /kaggle/working/yolo1024_stage1_bundle/data/yolo1024/manifests/train_stage1.txt
val: /kaggle/working/yolo1024_stage1_bundle/data/yolo1024/manifests/val.txt
names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis



# Cell 9 — Train stage 1

In [9]:
stage1_model = YOLO(MODEL_NAME)

stage1_model.train(
    data=BASE_YAML,
    epochs=EPOCHS_STAGE1,
    imgsz=YOLO_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    cache=False,
    amp=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=8e-4,
    weight_decay=1e-4,
    cos_lr=True,
    warmup_epochs=1.0,
    patience=2,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=4.0,
    translate=0.03,
    scale=0.08,
    fliplr=0.5,
    mosaic=0.22,
    mixup=0.02,
    close_mosaic=3,
    plots=False,
    save=True,
    project=RUN_DIR,
    name="stage1_base",
    verbose=True,
)

stage1_best = os.path.join(RUN_DIR, "stage1_base", "weights", "best.pt")
print("Stage 1 best:", stage1_best)

Ultralytics 8.4.42 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=3, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo1024_stage1_bundle/data/yolo1024/yolo1024_stage1.yaml, degrees=4.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=8, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0008, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.02, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.22, multi_scale=0.0, name=stage1_base, nbs=64, nms=False, opset=None, optimize=False, op

# Cell 10 — Validate, export, cleanup

In [10]:
stage1_ckpt = YOLO(stage1_best)
stage1_metrics = stage1_ckpt.val(data=BASE_YAML, imgsz=YOLO_SIZE, device=DEVICE)
print(stage1_metrics)

STAGE1_EXPORT = os.path.join(WORK_DIR, "yolov8m_1024_stage1.pt")
shutil.copy2(stage1_best, STAGE1_EXPORT)

summary = {
    "PREP_ROOT": PREP_ROOT,
    "COMP_ROOT": COMP_ROOT,
    "DATA_1024": DATA_1024,
    "stage1_export": STAGE1_EXPORT,
    "train_images": int(len(train_split)),
    "val_images": int(len(val_split)),
    "train_records": int(len(train_records_df)),
    "val_records": int(len(val_records_df)),
    "focus_classes": sorted(list(FOCUS_CLASS_IDS)),
    "rare_classes": sorted(list(RARE_CLASS_IDS)),
}
with open(os.path.join(WORK_DIR, "stage1_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

# keep output smaller
shutil.rmtree(RUN_DIR, ignore_errors=True)
gc.collect()
torch.cuda.empty_cache()

total_bytes = 0
for root, _, files in os.walk(WORK_DIR):
    for fn in files:
        total_bytes += os.path.getsize(os.path.join(root, fn))
print(f"Approx stage1 output size: {total_bytes / (1024**3):.3f} GB")

Ultralytics 8.4.42 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,847,866 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1772.5±357.5 MB/s, size: 101.8 KB)
val: Scanning /kaggle/working/yolo1024_stage1_bundle/data/yolo1024/labels/val/full.cache... 3000 images, 2121 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3000/3000 898.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 188/188 1.3s/it 3:59
                   all       3000       4739       0.35       0.36      0.305      0.152
    Aortic enlargement        638        716      0.647      0.838      0.833      0.516
           Atelectasis         34         41      0.213      0.195     0.0927      0.034
         Calcification         87        132      0.277      0.235      0.143     0.0562
          Cardiomegaly        477        498      0.723      0.878      0.877      0.55

In [11]:
import os
from IPython.display import FileLink

# 1. Zip your folder (the fastest compression method)
# Replace 'your_folder_name' with the actual folder you want to download
!zip -r -q output_data.zip /kaggle/working/

# 2. Generate and display the fast download link
FileLink(r'output_data.zip')

/kaggle/working/output_data.zip